In [ ]:
import pandas as pd
import requests
import json


In [ ]:
bazaar_data = requests.get('https://api.hypixel.net/v2/skyblock/bazaar').json()
items_in_bz = set()

bz_lst = []

for item in bazaar_data['products']:
  items_in_bz.add(item)
  temp = list(bazaar_data['products'][item]['quick_status'].values())
  temp[5] = format(float(temp[5]))
  temp[0] = temp[0].strip()
  bz_lst.append(temp)

column_names = list(bazaar_data['products']['IRON_INGOT']['quick_status'].keys())

df_bz = pd.DataFrame(columns = column_names, data = bz_lst)

df_bz = df_bz.sort_values('productId')
df_bz

,productId,sellPrice,sellVolume,sellMovingWeek,sellOrders,buyPrice,buyVolume,buyMovingWeek,buyOrders
545,ABSOLUTE_ENDER_PEARL,12302.215960,61781,433471,16,14521.765587529975,25101,201994,96
804,ACACIA_BIRDHOUSE,646219.338136,5824,630,279,847256.0893548387,14242,5383,915
1085,AGARIMOO_TONGUE,6208.409537,127145,1041163,21,7228.790087588121,213264,334381,349
939,ALLIGATOR_SKIN,531021.700000,3021,60263,18,566722.7,433,27429,22
284,AMALGAMATED_CRIMSONITE,0.000000,0,0,0,0.0,0,0,0
...,...,...,...,...,...,...,...,...,...
58,Y,55002.436738,27123,16089,39,87062.01245791247,11137,19598,85
603,YELLOW_FLOWER,86.046327,11988127,3078384,240,140.41638508128787,131074,832271,191
1152,YOGGIE,51.580704,6032695,970245,145,260.2747191852628,293987,335390,306
518,YOUNG_FRAGMENT,10870.960002,814363,1234883,140,13403.39270315091,117116,608042,630


In [ ]:
craft_prices = []

item_data = json.load(open('InternalNameMappings.json', 'r'))
for item in item_data:
  working = True
  if item not in items_in_bz:
    continue
  if 'recipe' in item_data[item]:
    recipe = item_data[item]['recipe']
    craft_price = 0
    for slot in recipe:
      materials = str(recipe[slot])
      if ':' not in materials:
        continue
      colon_loc = materials.index(':')
      item_id = materials[:colon_loc].strip()
      count = int(materials[colon_loc + 1:].strip())
      try:
        price = df_bz.loc[df_bz['productId'] == item_id, 'buyPrice'].values[0]
      except IndexError:
        working = False

      craft_price += float(price) * count
      buy_price = df_bz.loc[df_bz['productId'] == item, 'buyPrice'].values[0]
      sell_price = df_bz.loc[df_bz['productId'] == item, 'sellPrice'].values[0]
      diff = sell_price - craft_price
      diff_p = diff / craft_price
    if working:
      craft_prices.append([item, buy_price, sell_price, craft_price, diff, diff_p])

cols = ['id', 'buy price', 'sell price', 'craft price', 'profit', 'profit (%)']
flips_df = pd.DataFrame(columns = cols, data = craft_prices)


flips_df





<ipython-input-3-0ac73d304972>:27: RuntimeWarning: invalid value encountered in scalar divide
  diff_p = diff / craft_price
<ipython-input-3-0ac73d304972>:27: RuntimeWarning: divide by zero encountered in scalar divide
  diff_p = diff / craft_price


,id,buy price,sell price,craft price,profit,profit (%)
0,ABSOLUTE_ENDER_PEARL,14521.765587529975,1.230222e+04,1.879569e+04,-6.493472e+03,-0.345477
1,ACACIA_BIRDHOUSE,847256.0893548387,6.462193e+05,8.200606e+05,-1.738413e+05,-0.211986
2,AMALGAMATED_CRIMSONITE_NEW,143054.6,1.125939e+05,2.947813e+05,-1.821874e+05,-0.618043
3,AUTO_SMELTER,6184.200000000001,3.400391e+00,1.369674e+02,-1.335670e+02,-0.975174
4,BLAZE_WAX,581648.8181818182,3.677106e+05,4.080861e+05,-4.037545e+04,-0.098939
...,...,...,...,...,...,...
226,WHALE_BAIT,4212.997876498555,3.639604e+03,6.507665e+03,-2.868061e+03,-0.440720
227,WHEAT,12.5669363022809,3.900000e+00,0.000000e+00,3.900000e+00,inf
228,WHIPPED_MAGMA_CREAM,203905.27271557273,1.986144e+05,2.900366e+05,-9.142216e+04,-0.315209
229,WINTER_WATER_ORB,9460131.6,2.895431e+06,8.413610e+06,-5.518179e+06,-0.655863


In [ ]:
flips_df.sort_values('profit (%)')

,id,buy price,sell price,craft price,profit,profit (%)
164,HAY_BLOCK,0.0,0.000000,113.102427,-113.102427,-1.000000
62,ENCHANTED_INK_SACK,94999.59684210528,1.104105,111258.805296,-111257.701191,-0.999990
173,ICE_BAIT,2.0,0.200000,70.960088,-70.760088,-0.997182
86,ENCHANTED_PRISMARINE_CRYSTALS,392.10152920531465,339.743743,109370.509900,-109030.766157,-0.996894
201,PUMPKIN_BOMB,4326.7271929824565,2922.100141,457869.840956,-454947.740815,-0.993618
...,...,...,...,...,...,...
9,CARROT_BAIT,512.2506677505517,111.148568,69.283583,41.864985,0.604255
57,ENCHANTED_HARD_STONE,715.5989386614292,588.700000,229.925128,358.774872,1.560399
227,WHEAT,12.5669363022809,3.900000,0.000000,3.900000,inf
58,ENCHANTED_HAY_BLOCK,0.0,0.000000,0.000000,0.000000,NaN
